# More about functions as parameters

Lecture notes adapted from Alvin Alexander's *Functional Programming Simplified*

Functions can be defined as values as seen before:

In [1]:
val isEven = (i: Int) => i % 2 == 0

isEven: Int => Boolean = ammonite.$sess.cmd1$Helper$$Lambda/0x00007fcd642d5a48@32ad8d16

In [2]:
val sum = (x:Int,y:Int) => x+y 

sum: (Int, Int) => Int = ammonite.$sess.cmd2$Helper$$Lambda/0x00007fcd648fc8e8@1127a590

Then we can use functions that can take these functions as parameters, as is the case with `map` or `filter`:

In [3]:
val list = List.range(0, 10)

val evens = list.filter(isEven)


list: List[Int] = List(0, 1, 2, 3, 4, 5, 6, 7, 8, 9)
evens: List[Int] = List(0, 2, 4, 6, 8)

The key points of this are:
 - The `filter` method accepts a function as an input parameter.
 - The functions you pass into `filter` must match the type signature that filter
expects — in this case `Int => Boolean`

## How filter works

The Scaladoc shows the type of functions `filter` accepts:

```
def filter(p: (A) => Boolean): List[A]
```

 The Scaladoc text shows that `filter` takes a predicate, which is just a function that returns a `Boolean` value.

This part of the Scaladoc:
```
p: (A) => Boolean
```
means that filter takes a function input parameter which it names `p`, and `p` must transform a generic input `A` to a resulting `Boolean` value.


Because **`isEven`** has this type: `Int => Boolean`, it can be used with `filter`

 Without the `filter` method, we’d have to write a custom method like this to do the same work:
 

In [4]:
import scala.collection.mutable.ArrayBuffer

def getEvens(list: List[Int]): List[Int] = 
   val tmpArray = ArrayBuffer[Int]()
   for (elem <- list) do
     if (elem % 2 == 0) then tmpArray += elem
   tmpArray.toList
 

import scala.collection.mutable.ArrayBuffer


defined function getEvens

In [5]:
val result = getEvens(list)

result: List[Int] = List(0, 2, 4, 6, 8)

 Compare all of that imperative code to this equivalent functional code:

In [6]:
 val result = list.filter(_ % 2 == 0)

result: List[Int] = List(0, 2, 4, 6, 8)

## Defining functions that take other functions

To define a function that takes another function as an input parameter, all you have to do is define the signature of the function you want to accept.

*Example:* a function `sayHello` takes a function `callback` as an input parameter. 

It has no input parameters returns nothing:

In [7]:
def sayHello(callback: () => Unit) =
  callback()

defined function sayHello

 - `callback` is the name of the function parameter
 - the callback signature specifies the type of function
 - `()` means it takes no input parameters.
 - `Unit` indicates the function returns nothing.
 - When `sayHello` is called, the `callback()` line invokes the function passed in.

Now that `sayHello` is defined we can create a function to match `callback`’s signature

This function matches the type signature of `callback`:

In [8]:
def helloJimmy(): Unit = 
  println("Hello, Jimmy") 

defined function helloJimmy

In [9]:
helloJimmy _

res9: () => Unit = ammonite.$sess.cmd9$Helper$$Lambda/0x00007fcd64957038@38b5df6d

And now we can use it:

In [10]:
sayHello(helloJimmy)

Hello, Jimmy


In [11]:
def helloMary() = println("Hi Mary!!!")

defined function helloMary

In [12]:
sayHello(helloMary)

Hi Mary!!!


In [13]:
def countToTen() = 
  for i <- (1 to 10) do println(i)

defined function countToTen

In [14]:
sayHello(countToTen)

1
2
3
4
5
6
7
8
9
10


In [15]:
def helloSam(a:Int):Unit = println(a)


defined function helloSam

In [15]:
sayHello(helloSam)

-- [E007] Type Mismatch Error: cmd16.sc:1:21 -----------------------------------
1 |val res16 = sayHello(helloSam)
  |                     ^^^^^^^^
  |      Found:    Int => Unit
  |      Required: () => Unit
  |
  |      The following import might make progress towards fixing the problem:
  |
  |        import sourcecode.Text.generate
  |
  |
  | longer explanation available when compiling with `-explain`
Compilation Failed

## More parameters

Next, here’s a function named `executeAndPrint` that takes a function and two Int parameters, and returns nothing. 

In [16]:
 def executeAndPrint(f: (Int, Int) => Int, x: Int, y: Int): Unit = 
   val result = f(x, y)
   println(result)

defined function executeAndPrint

To demonstrate `executeAndPrint`, here are a couple of functions take two `Int` parameters and return an `Int`:

In [17]:
def sum(x: Int, y: Int) = x + y

def multiply(x: Int, y: Int) = x * y

def complex(x: Int, y: Int) = x*x + y*x + 5*y 

defined function sum
defined function multiply
defined function complex

In [18]:
executeAndPrint(sum,3,4)
executeAndPrint(multiply,3,4)
executeAndPrint(complex,3,4)

7
12
41


## And more functions as parameters

We can also take more than one function as parameters:

In [19]:
 def execTwoFunctions(f1: (Int, Int) => Int,
                      f2: (Int, Int) => Int,
                      a: Int,
                      b: Int): Tuple2[Int, Int] = 
   val result1 = f1(a, b)
   val result2 = f2(a, b)
   (result1, result2)

defined function execTwoFunctions

In [20]:
execTwoFunctions(sum, complex, 4, 5) 

res20: (Int, Int) = (9, 61)

## Dissecting the map function

For the sake of learning, let's assume that we don't have a bulit-in `map` function.

We see how we can implement the map function, i.e. apply other functions to each element in a `List`.


### Type singature

We can start with the method signature. It should of course be named `map`:

```
def map
```

For the sake of simlicity, le'ts restrict it to a list of `Int` values.
Then, `map` should take a function as input and a ```List[Int]```:

```
 def map(f: (?) => ?, list: List[Int]): ???
```


Knowing how map works, it should return a List with the same number of elements of the input List. 

What are the elements of this new list?
`map` applies a function to every element in the input list, so the output List can be anything: a `List[Double]`, `List[Float]`, `List[Foo]`,
 etc. Therefore the List needs to contain a generic type, e.g., `[A]`:

``` 
 def map[A](f: (?) => ?, list: List[Int]): List[A]
```


Now we can focus on the function `f`. It should follow the following conditions:
 - Because `f`’s input come from the `List[Int]`, its input should be `Ìnt`
 - Because the `map` function returns a `List[A]` the `f` must also return type `A`
 
Therefore the type signature should be:
```
 def map[A](f: Int => A, list: List[Int]): List[A]
``` 

### Body of the map function

A `map` function works on every element in a list, so we can use a `for` loop to iterate over every element in the input list.

```
 def map[A](f: (Int) => A, list: List[Int]): List[A] = 
   for (x <- list)  yield ???
```

Now, what should the loop yield?

 The `for` loop should yield the result of applying the input function `f` to the current element in the loop. 
 
 Therefore, we cancomplete the `map` body as: 

In [21]:
def map[A](f: (Int) => A, list: List[Int]): List[A] = 
  for x <- list yield 
    f(x) //<-- apply 'f' to each element 'x'
 

defined function map

Now we can use our new ``map` function:

In [2]:
def double(i: Int): Int = i * 2

def classify(i: Int): String = if i<5 then s"small number $i" else s"big number $i"

defined function double
defined function classify

In [3]:
val nums = (1 to 10).toList

nums: List[Int] = List(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)

In [4]:
map(double,nums)

res4: List[Int] = List(2, 4, 6, 8, 10, 12, 14, 16, 18, 20)

In [5]:
map(classify,nums)

res5: List[String] = List(
  "small number 1",
  "small number 2",
  "small number 3",
  "small number 4",
  "big number 5",
  "big number 6",
  "big number 7",
  "big number 8",
  "big number 9",
  "big number 10"
)

### Making it generic

This map was defined for a lsit of `Int`. We can make it generic for a list of any type, e.g., `B`:

In [6]:
def map[A,B](f: (B) => A, list: List[B]): List[A] = 
  for x <- list yield 
    f(x) //<-- apply 'f' to each element 'x'
 

defined function map

In [7]:
map(classify,nums)

res7: List[String] = List(
  "small number 1",
  "small number 2",
  "small number 3",
  "small number 4",
  "big number 5",
  "big number 6",
  "big number 7",
  "big number 8",
  "big number 9",
  "big number 10"
)

In [8]:
val temperatures = List(32.3,4.5,26.5)

map( (d:Double) => if d>20 then "hot" else "cold" , temperatures)

temperatures: List[Double] = List(32.3, 4.5, 26.5)
res8_1: List[String] = List("hot", "cold", "hot")

## By-Name Parameters

Typically, parameters to functions are **by-value** parameters; that is, the value of the parameter is determined before it is passed to the function

For example in the example below:

In [9]:
 case class Person(var name: String)

defined class Person

In [10]:
def changeName(p: Person) = 
  p.name = "Al"

defined function changeName

In [11]:
val lucas = Person("lucas")

lucas: Person = Person(name = "lucas")

The parameter to the `changeName` method is like a pointer to the `Person` instance

In [12]:
changeName(lucas)
lucas

res12_1: Person = Person(name = "Al")

### Using by-name parameters

Using by-name parameters indicates that the argument is not evaluated at the point of function application, but instead is **evaluated at each use** within the function

**Example: `timer` function**
Imagine we want a timer function that executes some function but also measures how long it took.

Using this function we could call it like this:

```
    val (result, time) = timer(someLongRunningAlgorithm)

```

or even like this:

```
val (result, time) = timer {
 ...
 some code here
 ...
 }
```

We can try to start with an example of a code we want to time, like the following calculation: 

In [13]:
def calculate(x:Int,y:Int) = 
  println("start computing")
  x*x +5*x*y

defined function calculate

If we want to time it we could do it as follows:

In [14]:
 val startTime = System.nanoTime
 val result = calculate(4,5)
 val stopTime = System.nanoTime
 val delta = stopTime- startTime
 (result, delta/1000000d)

start computing


startTime: Long = 13430198299808L
result: Int = 116
stopTime: Long = 13430198631218L
delta: Long = 331410L
res14_4: (Int, Double) = (116, 0.33141)

We can now take the timing parts and put them into a method `timer`, where the function is passed as a parameter:

In [15]:
def timer(f:(Int,Int)=>Int,x:Int,y:Int) =
  val startTime = System.nanoTime
  val result = f(x,y)  // here is the invocation
  val stopTime = System.nanoTime
  val delta = stopTime- startTime
  (result, delta/1000000d)

defined function timer

In [16]:
timer(calculate,4,5)

start computing


res16: (Int, Double) = (116, 0.765355)

In [17]:
def calculate2(d:Int,i:Int):Double = d.toDouble/i


defined function calculate2

However this is not what we really want, this wll only work for a function with two `Int` params, otherwise it's not applicable:

In [17]:

timer(calculate2,4,5)

-- [E007] Type Mismatch Error: cmd18.sc:1:18 -----------------------------------
1 |val res18 = timer(calculate2,4,5)
  |                  ^^^^^^^^^^
  |      Found:    Double
  |      Required: Int
  |
  |      The following import might make progress towards fixing the problem:
  |
  |        import sourcecode.Text.generate
  |
  |
  | longer explanation available when compiling with `-explain`
Compilation Failed

One way to try to fix it is to change it to a generic type:

In [18]:
def timer[A](f:(Int,Int)=>A,x:Int,y:Int) =
  val startTime = System.nanoTime
  val result = f(x,y)  // here is the invocation
  val stopTime = System.nanoTime
  val delta = stopTime- startTime
  (result, delta/1000000d)

defined function timer

In [19]:
timer(calculate2,4,5)

res19: (Double, Double) = (0.8, 0.018864)

However this is still too limited by the input parameters, which are always two `Int`

In [19]:
timer( (i:Int) => i*4, 3 )

-- [E007] Type Mismatch Error: cmd20.sc:1:19 -----------------------------------
1 |val res20 = timer( (i:Int) => i*4, 3 )
  |                   ^^^^^^^^^^^^^^
  |      Found:    Int => Int
  |      Required: (Int, Int) => Int
  |
  |      The following import might make progress towards fixing the problem:
  |
  |        import sourcecode.Text.generate
  |
  |
  | longer explanation available when compiling with `-explain`
Compilation Failed

The alternative to this is to use Scala's call by-name, whose syntax is as follows:

```
 def timer(blockOfCode: => theReturnType) ...

```

Changing this to our timer code, it  becomes:

In [20]:
def timer[A](blockOfCode: =>A) =
  val startTime = System.nanoTime
  val result = blockOfCode  // here is the code
  val stopTime = System.nanoTime
  val delta = stopTime- startTime
  (result, delta/1000000d)

defined function timer

In [21]:
timer(calculate2(3,4))

res21: (Double, Double) = (0.75, 0.016134)

In [22]:
timer {
  val i=2 
  val j= calculate(i,4)
  calculate2(i,j)
}

start computing


res22: (Double, Double) = (0.045454545454545456, 0.296472)

### When is the code running?

In the case of the timer function, it executes the blockOfCode when the second line of the function is reached. 
 
But we can also see it in this example: 

In [23]:
def test[A](codeBlock: => A) = 
  println("before 1st codeBlock")
  val a = codeBlock
  println(a)
  Thread.sleep(10)
 
  println("before 2nd codeBlock")
  val b = codeBlock
  println(b)
  Thread.sleep(10)
 
  println("before 3rd codeBlock")
  val c = codeBlock
  println(c)
 

defined function test

As you can see in the example the code is called each time:

In [28]:
import scala.util.Random

test(Random.nextInt(5))

before 1st codeBlock
3
before 2nd codeBlock
0
before 3rd codeBlock
2


import scala.util.Random



We can see it clearly with a timestamp:

In [29]:
test {
    (System.currentTimeMillis,
    Random.nextInt(5))
}

before 1st codeBlock
(1774266778071,4)
before 2nd codeBlock
(1774266778082,0)
before 3rd codeBlock
(1774266778092,0)
